# Crypto + 60/40 with portfolio vol targeting

Take a small BTC sleeve, bolt it onto a vanilla 60/40, and then scale the whole portfolio to a fixed annualized vol target. The point is to compare:

- pure 60/40 (SPY/AGG)
- 60/40 + a static 5% / 10% BTC sleeve (allocation taken from SPY)
- the same blended portfolio with **vol targeting** (~10% annualized), capping leverage at 1.5x

Vol targeting is the only non-trivial overlay; it uses the helper in `alpha_lab.portfolio.vol_target`.

In [ ]:
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns
from alpha_lab.portfolio.vol_target import apply_vol_target
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly

UNIVERSE = ["SPY", "AGG", "BTC-USD"]
START = "2015-01-01"           # BTC-USD on yfinance goes back to 2014; allow 1y burn-in
END = None

BTC_SLEEVE = 0.10               # 10% allocation, taken from SPY
TARGET_VOL = 0.10               # 10% annualized portfolio vol
VOL_LOOKBACK = 63               # ~3 months
MAX_LEV = 1.5
REBAL = "ME"
COSTS_BPS = 2.0                 # blended (ETF ~1, BTC ~5)
SLIPPAGE_BPS = 4.0

In [ ]:
prices = load_prices(UNIVERSE, start=START, end=END).dropna(how="all").ffill().dropna()
prices.tail()

In [ ]:
# Static target weights, ffilled to daily. run_backtest does the 1-day lag.
def static_weights(target: dict[str, float]) -> pd.DataFrame:
    w = pd.DataFrame(0.0, index=prices.index, columns=UNIVERSE)
    for k, v in target.items():
        w[k] = v
    return w

w_6040 = static_weights({"SPY": 0.6, "AGG": 0.4})
w_blend = static_weights({"SPY": 0.6 - BTC_SLEEVE, "AGG": 0.4, "BTC-USD": BTC_SLEEVE})

# Vol-targeted version: scale the blended weights so realized portfolio vol ~= TARGET_VOL.
asset_rets = prices.pct_change().fillna(0.0)
w_blend_vt = apply_vol_target(
    w_blend,
    asset_rets,
    target_vol=TARGET_VOL,
    lookback_days=VOL_LOOKBACK,
    max_leverage=MAX_LEV,
).fillna(0.0)
w_blend_vt.tail()

In [ ]:
res_6040 = run_backtest(w_6040, prices, rebalance=REBAL, costs_bps=COSTS_BPS, slippage_bps=SLIPPAGE_BPS)
res_blend = run_backtest(w_blend, prices, rebalance=REBAL, costs_bps=COSTS_BPS, slippage_bps=SLIPPAGE_BPS)
res_blend_vt = run_backtest(w_blend_vt, prices, rebalance=REBAL, costs_bps=COSTS_BPS, slippage_bps=SLIPPAGE_BPS)

streams = pd.DataFrame({
    "60/40": res_6040.returns,
    f"60/40 + {BTC_SLEEVE:.0%} BTC": res_blend.returns,
    f"60/40 + {BTC_SLEEVE:.0%} BTC, vol-target {TARGET_VOL:.0%}": res_blend_vt.returns,
    "BH BTC": asset_rets["BTC-USD"],
    "BH SPY": asset_rets["SPY"],
}).dropna(how="all")
pd.DataFrame({name: pd.Series(summary(s)) for name, s in streams.items()})

In [ ]:
# What does the vol-target leverage look like over time?
leverage = w_blend_vt.abs().sum(axis=1)
leverage.describe().rename("leverage_summary")

In [ ]:
leverage.plot(title="Portfolio gross leverage (vol-targeted blend)")

In [ ]:
annual_turnover_vt = res_blend_vt.turnover.resample("YE").sum()
annual_costs_vt = res_blend_vt.costs.resample("YE").sum()
pd.DataFrame({"turnover_per_year": annual_turnover_vt, "cost_drag_per_year": annual_costs_vt})

In [ ]:
equity_curve(streams)

In [ ]:
drawdown_chart(res_blend_vt.returns)

In [ ]:
heatmap_monthly(res_blend_vt.returns)

## Read

- A static small-BTC sleeve usually wins on CAGR over the past decade but pays for it in drawdowns (BTC crashes drag the whole portfolio).
- Vol targeting flattens those crashes by de-leveraging into stress and re-leveraging into calm — at the cost of higher turnover.
- Sensitivities to test: `BTC_SLEEVE` (5% vs 10% vs 20%), `TARGET_VOL` (8/10/12%), and `MAX_LEV` (cap at 1.0 = no margin).
- Implementation: ETFs in brokerage + BTC in Coinbase/Kraken. Monthly rebalance. Vol-target leverage>1 requires margin or futures — set `MAX_LEV=1.0` for a fully-funded retail version.